In [4]:
# Separate the features and target variable
X = qual.drop(['Target'], axis=1)  
y = qual['Target']

In [5]:
# Split data into train, test, and validation sets first
X_train_full, X_temp, y_train_full, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Separate the positive and negative classes in the training set
positive_class = X_train_full[y_train_full == 1]
negative_class = X_train_full[y_train_full == 0]
positive_class_labels = y_train_full[y_train_full == 1]
negative_class_labels = y_train_full[y_train_full == 0]

# Downsample negative class to 250,000 instances 
n_negative_samples = min(250000, len(negative_class))
negative_class_sampled = negative_class.sample(n=n_negative_samples, random_state=42)
negative_class_labels_sampled = negative_class_labels.loc[negative_class_sampled.index]

# Combine positive class with the sampled negative class
X_train_new = pd.concat([positive_class, negative_class_sampled], axis=0).reset_index(drop=True)
y_train_new = pd.concat([positive_class_labels, negative_class_labels_sampled], axis=0).reset_index(drop=True)

# Shuffle the new training set
X_train_new, y_train_new = shuffle(X_train_new, y_train_new, random_state=42)


In [6]:
# Identify categorical and numerical columns
categorical_columns = X_train_new.select_dtypes(include=['object']).columns
numerical_columns = X_train_new.select_dtypes(include=['float64', 'int64']).columns

# Target Encoding
class TargetEncoder:
    def __init__(self, smoothing=1.0):
        self.smoothing = smoothing
        self.global_mean = None
        self.category_means = {}
    
    def fit(self, X_train, y_train, columns):
        self.global_mean = y_train.mean()
        for col in columns:
            stats = y_train.groupby(X_train[col]).agg(['mean', 'count'])
            means = stats['mean']
            counts = stats['count']
            smooth = 1 / (1 + np.exp(-(counts - 1) / self.smoothing))
            self.category_means[col] = self.global_mean * (1 - smooth) + means * smooth

    def transform(self, X):
        X_encoded = X.copy()
        for col in self.category_means:
            X_encoded[col] = X_encoded[col].map(self.category_means[col]).fillna(self.global_mean)
        return X_encoded
    
    def fit_transform(self, X_train, y_train, columns):
        self.fit(X_train, y_train, columns)
        return self.transform(X_train)

# Initialize and apply target encoder
target_encoder = TargetEncoder(smoothing=1.0)
X_train_encoded = target_encoder.fit_transform(X_train_new, y_train_new, columns=categorical_columns)
X_test_encoded = target_encoder.transform(X_test)
X_val_encoded = target_encoder.transform(X_val)

In [7]:
# Apply SelectKBest for feature selection on the training set
k_best = 248
select_k_best = SelectKBest(mutual_info_classif, k=k_best)
X_train_selected = select_k_best.fit_transform(X_train_encoded, y_train_new)

# Apply the same feature selection to the test and validation sets
X_test_selected = select_k_best.transform(X_test_encoded)
X_val_selected = select_k_best.transform(X_val_encoded)


In [8]:
# Resampling strategies
resampling_strategy = {
    'minority': 0.05,  # Oversample the minority class to 5% of the dataset
    'majority': 0.8    # Undersample the majority class to 80% of the dataset
}

# Pipeline with SMOTE and RandomUnderSampler
pipeline = ImbalancedPipeline([
    ('oversample', SMOTE(sampling_strategy=resampling_strategy['minority'], n_jobs=-1)),
    ('undersample', RandomUnderSampler(sampling_strategy=resampling_strategy['majority']))
])

# Fit and resample the training data
X_train_resampled, y_train = pipeline.fit_resample(X_train_selected
                                                             , y_train_new)

# Check the class distribution after resampling
class_distribution_after = pd.Series(y_train).value_counts(normalize=True)
print("Class distribution after resampling:", class_distribution_after)


  File "C:\Users\Katerina\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Katerina\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Katerina\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Katerina\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


Class distribution after resampling: Target
0    0.555556
1    0.444444
Name: proportion, dtype: float64


In [9]:
# Apply RobustScaler on the training set
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)

# Apply the same scaler to the test and validation sets
X_test_scaled = scaler.transform(X_test_selected)
X_val_scaled = scaler.transform(X_val_selected)